In [3]:
import numpy as np
import pandas as pd
import os
import librosa
import matplotlib.pyplot as plt
import IPython
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Reshape, MaxPooling2D, Dropout, Conv2D, MaxPool2D, Flatten
from tensorflow.keras.utils import to_categorical

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import os

paths = []
labels = []

# Define the root directory (Updated to your specific path)
real_root_dir = r'C:\Users\CRESCENT\Desktop\Audio-DeepFake-Detection\input\Real audio'
fake_root_dir = r'C:\Users\CRESCENT\Desktop\Audio-DeepFake-Detection\input\Fake audio'

# Iterate through the subdirectories
for filename in os.listdir(real_root_dir):
    file_path = os.path.join(real_root_dir, filename)
    paths.append(file_path)
    labels.append('real')
    
for filename in os.listdir(fake_root_dir):
    file_path = os.path.join(fake_root_dir, filename)
    paths.append(file_path)
    labels.append('fake')

print('Dataset is loaded')

In [ ]:
df = pd.DataFrame({
    'path': paths,
    'label': labels
})

df.head()

In [ ]:
df['label'].value_counts()

In [ ]:
audio_path = df['path'][0]
audio, sr = librosa.load(audio_path)
librosa.display.waveshow(audio, sr=sr)
plt.show()

In [ ]:
def extract_mel(file_path):
    audio, sr = librosa.load(file_path)
    mel = librosa.feature.melspectrogram(y=audio, sr=sr)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

In [ ]:
features = []
for path in df['path']:
    features.append(extract_mel(path))

In [ ]:
X = np.array(features)
y = np.array(df['label'])

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_cat = to_categorical(y_encoded)

In [ ]:
X = X[..., np.newaxis]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42
)

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=X_train.shape[1:]),
    MaxPool2D((2,2)),
    Dropout(0.3),

    Conv2D(64, (3,3), activation='relu'),
    MaxPool2D((2,2)),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(2, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential, regularizers
with strategy.scope():
    # Create and compile the model inside the strategy scope
    model = Sequential([
        layers.Reshape((40, 500, 1), input_shape=xtrain.shape[1:]),
        layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Reshape((-1, 128)),
        layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)),
        layers.BatchNormalization(),
        layers.Bidirectional(layers.LSTM(128, dropout=0.3, recurrent_dropout=0.2)),
        layers.BatchNormalization(),
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    

# Print the model summary to confirm the structure
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_accuracy',        
    patience=10,                    
    restore_best_weights=True,     
    mode='max',                     
    verbose=1                      
)

history = model.fit(
    xtrain, ytrain,                 
    epochs=100,                     
    batch_size=32,                  
    validation_data=(xtest, ytest),   
    callbacks=[early_stopping]      
)

In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

In [ ]:
loss,accuracy = model.evaluate(xtest,ytest)

In [ ]:
cpu_model = tf.keras.models.clone_model(model)

cpu_model.set_weights(model.weights)

cpu_model.save('my_model.h5')